<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# 5G Network Operator Agent with NVIDIA NIM

This notebook demonstrates an educational decision loop over deterministic synthetic observations. It does not train a model, connect to a live RAN or other physical infrastructure, or claim physical-network fidelity.

## Outcome and prerequisites

Follow the [README](README.md) to create the Python environment. The local noop and scripted-relief baselines always run; the hosted NVIDIA NIM policy is optional and requires an existing `NVIDIA_API_KEY`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

def find_example_dir():
    candidates = (Path.cwd(), Path.cwd() / "community" / "5g-network-operator-agent")
    for candidate in candidates:
        if (candidate / "network_environment.py").is_file():
            return candidate
    raise RuntimeError("Run this notebook from the example directory or repository root.")

EXAMPLE_DIR = find_example_dir()
if str(EXAMPLE_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLE_DIR))
load_dotenv(dotenv_path=EXAMPLE_DIR / ".env", override=False)

from network_environment import (
    NetworkEnvironment,
    default_scenario,
    noop_policy,
    run_episode,
    scripted_relief_policy,
    tool_schemas,
)

def ue_frame(observation):
    return pd.DataFrame([ue.to_dict() for ue in observation.ues]).set_index("ue_id")

def tool_frame(tools):
    return pd.DataFrame(
        {
            "action": tool["function"]["name"],
            "description": tool["function"]["description"],
            "parameters": json.dumps(tool["function"]["parameters"], sort_keys=True),
        }
        for tool in tools
    )

def result_frame(episodes):
    return pd.DataFrame(
        {"policy": name, "turns": len(episode.transitions), "total_score": episode.total_reward}
        for name, episode in episodes.items()
    ).set_index("policy")

def plot_cumulative_scores(episodes):
    _, axis = plt.subplots(figsize=(7, 4))
    for name, episode in episodes.items():
        cumulative = 0.0
        values = [0.0]
        for transition in episode.transitions:
            cumulative += transition.reward["total"]
            values.append(cumulative)
        axis.plot(range(len(values)), values, marker="o", label=name)
    axis.set(xlabel="turn", ylabel="cumulative score", title="Same synthetic scenario and four-turn horizon")
    axis.legend()
    axis.grid(alpha=0.25)
    plt.show()


In [ ]:
environment = NetworkEnvironment(default_scenario())
initial_observation = environment.reset()
display(ue_frame(initial_observation))
display(pd.DataFrame([initial_observation.cell.to_dict()], index=["cell"]))
with pd.option_context("display.max_colwidth", None):
    display(tool_frame(tool_schemas()))


## Transparent scoring

Every score term is non-positive and zero is ideal. Compare returns only on the same bundled scenario and four-turn horizon. Scripted relief is a fixed local rule, not a learned policy.

In [ ]:
MAX_STEPS = 4
scenario = default_scenario()
episodes = {
    "noop": run_episode(noop_policy, scenario=scenario, max_steps=MAX_STEPS),
    "scripted_relief": run_episode(scripted_relief_policy, scenario=scenario, max_steps=MAX_STEPS),
}
display(result_frame(episodes))
plot_cumulative_scores(episodes)


## Optional hosted NVIDIA NIM policy

Set `NVIDIA_API_KEY` to enable the hosted policy; `NVIDIA_MODEL` optionally overrides the default. Without a key, only hosted evaluation is skipped.

In [ ]:
api_key = os.getenv("NVIDIA_API_KEY")
model_id = os.getenv("NVIDIA_MODEL") or "nvidia/nemotron-3-super-120b-a12b"
client = None
if not api_key:
    print("NVIDIA_API_KEY is not set; hosted-policy evaluation is skipped.")
else:
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=api_key)


## Strict one-call contract

The policy must return exactly one valid function call. Malformed output or request errors become explicit rejected actions with a finite penalty; they are never replaced by `noop`.

In [ ]:
def _invalid_model_action(reason):
    return {"name": f"__invalid_model_output__:{reason}", "arguments": {}}

def _parse_tool_call_response(response):
    try:
        choices = list(response.choices)
    except (AttributeError, TypeError):
        return _invalid_model_action("unexpected_choice_count")
    if len(choices) != 1:
        return _invalid_model_action("unexpected_choice_count")

    try:
        tool_calls = list(choices[0].message.tool_calls or [])
    except (AttributeError, TypeError):
        return _invalid_model_action("missing_tool_call")
    if not tool_calls:
        return _invalid_model_action("missing_tool_call")
    if len(tool_calls) != 1:
        return _invalid_model_action("multiple_tool_calls")

    call = tool_calls[0]
    function = getattr(call, "function", None)
    name = getattr(function, "name", None)
    arguments = getattr(function, "arguments", None)
    if getattr(call, "type", None) != "function" or not isinstance(name, str) or not name:
        return _invalid_model_action("non_function_call")
    if not isinstance(arguments, str):
        return _invalid_model_action("invalid_json_arguments")
    try:
        decoded_arguments = json.loads(arguments)
    except (TypeError, ValueError, RecursionError):
        return _invalid_model_action("invalid_json_arguments")
    if not isinstance(decoded_arguments, dict):
        return _invalid_model_action("non_object_arguments")
    return {"name": name, "arguments": decoded_arguments}

def _request_error_action(error):
    error_class = "".join(
        character.lower() if character.isalnum() else "_"
        for character in type(error).__name__
    )[:48].strip("_") or "exception"
    return _invalid_model_action(f"request_error_{error_class}")

def nim_policy(observation, tools):
    messages = [
        {
            "role": "system",
            "content": (
                "For this deterministic synthetic four-turn episode, select one bounded action per turn "
                "to maximize the cumulative score by bringing it closer to zero. Zero is ideal and "
                "more-negative is worse. Each after-state score is the sum of these costs: "
                "-0.45 * unmet_ratio; -0.20 * (1 - jain_fairness); "
                "-0.25 * sla_violations / ue_count; "
                "-0.10 * max(0, (prb_util_pct - 85) / 15); and "
                "-0.25 if the action is rejected. Use the current observation and remember that accepted "
                "controls persist into later turns. Select exactly one provided tool. "
                "Do not invent tools or return prose instead of a tool call."
            ),
        },
        {"role": "user", "content": json.dumps(observation.to_dict(), sort_keys=True)},
    ]
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            tools=tools,
            tool_choice="required",
            parallel_tool_calls=False,
            n=1,
            stream=False,
        )
    except Exception as error:
        return _request_error_action(error)
    return _parse_tool_call_response(response)


## Episode comparison and transition inspection

If configured, the hosted policy runs on the same fixed scenario and is added to the comparison. The detailed view then shows one complete before/action/after transition and its score terms.

In [ ]:
hosted_episode = None
if client is not None:
    hosted_episode = run_episode(nim_policy, scenario=scenario, max_steps=MAX_STEPS)
    episodes["hosted_nim"] = hosted_episode
    display(result_frame(episodes))
    plot_cumulative_scores(episodes)
    display(pd.DataFrame(
        {"accepted": transition.accepted, "error": transition.error}
        for transition in hosted_episode.transitions
    ))
selected_episode = hosted_episode or episodes["scripted_relief"]
selected_transition = selected_episode.transitions[0]
print("Before")
display(pd.DataFrame([selected_transition.before.cell.to_dict()], index=["cell"]))
display(ue_frame(selected_transition.before))
print("Action / validation")
display(pd.DataFrame([{
    "action": json.dumps(selected_transition.action, sort_keys=True),
    "accepted": selected_transition.accepted,
    "error": selected_transition.error,
}]))
print("After")
display(pd.DataFrame([selected_transition.after.cell.to_dict()], index=["cell"]))
display(ue_frame(selected_transition.after))
display(pd.DataFrame(
    {"score_term": name, "value": value}
    for name, value in selected_transition.reward.items()
))


## Limitations and next steps

This fixed synthetic example does not establish live-network performance or learned-policy improvement. Try another bounded action or hosted policy, then inspect the resulting transition. See the [NVIDIA API documentation](https://docs.api.nvidia.com/) and [`network_environment.py`](network_environment.py).